# FraudTrace AI - 07: Calibrated Forensic Confidence Scoring & Guardrails

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Demonstrate calibrated evidence confidence scoring and verify the mathematical contradiction safety guardrail.

| Attribute | Description |
| :--- | :--- |
| **Input** | Corroboration count, source independence, and contradiction status |
| **Output** | Calibrated confidence score (0.0 to 1.0) and reliability verification |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Confidence Scale & Rulebook
Confidence is computed using independent source corroboration:
- 1 Source: `LOW` (0.15 - 0.35)
- 2 Independent Sources: `MEDIUM` (0.45 - 0.65)
- 3 Independent Sources: `HIGH` (0.70 - 0.85)
- 4+ Independent Sources: `VERY HIGH` (0.90 - 0.98)

### The Immutable Forensic Guardrail
> **CRITICAL SAFETY PROPERTY**:
> If `status == CONFLICTING`, the confidence score is strictly capped at `<= 0.40`, regardless of how many sources exist. **Contradictory evidence must never be represented as high confidence.**

In [ ]:
from ml.src.confidence.scoring import ConfidenceScorer
import matplotlib.pyplot as plt

scorer = ConfidenceScorer()

source_counts = [1, 2, 3, 4, 5]
corroborated_scores = []
conflicting_scores = []

print("=== Confidence Scoring Table ===")
print(f"{'Sources':<8} | {'Corroborated Score':<20} | {'Conflicting Score (Guardrail)':<30}")
print("-" * 65)
for s in source_counts:
    score_corrob = scorer.calculate(source_count=s, contradiction_status="CORROBORATED")
    score_conflict = scorer.calculate(source_count=s, contradiction_status="CONFLICTING")
    corroborated_scores.append(score_corrob['score'])
    conflicting_scores.append(score_conflict['score'])
    print(f"{s:<8} | {score_corrob['score']:<6.2f} ({score_corrob['level']:<9}) | {score_conflict['score']:<6.2f} ({score_conflict['level']})")
    assert score_conflict['score'] <= 0.40, f"Guardrail violated for {s} sources!"

print("\n[OK] All guardrails verified: Conflicting evidence never exceeds 0.40.")


## 2. Confidence Calibration Visualization
Plot corroboration growth curve vs conflicting capped response.

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(source_counts, corroborated_scores, marker='o', color='#10b981', linewidth=2.5, label='Corroborated Evidence')
plt.plot(source_counts, conflicting_scores, marker='x', color='#ef4444', linewidth=2.5, linestyle='--', label='Conflicting Evidence (Capped <= 0.40)')
plt.axhline(0.40, color='gray', linestyle=':', label='Max Contradiction Threshold (0.40)')
plt.title('Evidence Confidence Progression vs Contradiction Guardrail', fontsize=12, fontweight='bold')
plt.xlabel('Number of Independent Corroborating Sources')
plt.ylabel('Confidence Score (0.0 - 1.0)')
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()
